# 🏪 Locaux Commerciaux — Location Marrakech
## Nettoyage, Feature Engineering & Modélisation XGBoost

**Objectif** : prédire le loyer mensuel (MAD) d'un local commercial à Marrakech.  
**Pipeline** : `pipeline/locations/pip_locaux.py`  
**Données** : `data/marrakech_immo_location/locaux_location.csv`

---


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "locations" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score

from pipeline.locations.pip_locaux import (
    load_data, split_and_encode, build_pipeline, train, evaluate,
    tune_hyperparams, plot_results, predict_price,
    DATA_PATH, MODEL_PATH,
    NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES, TARGET_LOG,
)

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print("✅ Imports OK")
print(f"   DATA  : {DATA_PATH}")
print(f"   MODEL : {MODEL_PATH}")


## 1. Chargement & Aperçu des données brutes

In [ ]:
import pandas as pd
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape brut : {df_raw.shape}")
df_raw.head(3)


In [ ]:
print("Colonnes disponibles :")
print(df_raw.columns.tolist())
print(f"\nNulls par colonne :")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0].to_string())


In [ ]:
print("Distribution type_bien :")
print(df_raw["type_bien"].value_counts().to_string())
print("\nDistribution quartier :")
print(df_raw["quartier"].value_counts().head(15).to_string())


### 1.1 Distribution prix bruts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Prix brut
valid_prix = df_raw["prix_num"].dropna()
axes[0].hist(valid_prix[valid_prix < 100_000], bins=50, color="#2563eb", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("Loyer (MAD/mois)"); axes[0].set_title("Distribution prix bruts (< 100K)")
axes[0].axvline(valid_prix.median(), color="red", ls="--", label=f"Médiane: {valid_prix.median():,.0f}")
axes[0].legend()

# Surface brute
valid_surf = df_raw["surface_num"].dropna()
axes[1].hist(valid_surf[valid_surf < 2000], bins=50, color="#10b981", alpha=0.8, edgecolor="white")
axes[1].set_xlabel("Surface (m²)"); axes[1].set_title("Distribution surface brute (< 2000 m²)")
axes[1].axvline(valid_surf.median(), color="red", ls="--", label=f"Médiane: {valid_surf.median():.0f} m²")
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Prix brut  — médiane: {valid_prix.median():,.0f} MAD | min: {valid_prix.min():,.0f} | max: {valid_prix.max():,.0f}")
print(f"Surface brute — médiane: {valid_surf.median():.0f} m² | min: {valid_surf.min():.0f} | max: {valid_surf.max():.0f}")


## 2. Nettoyage & Feature Engineering (via pipeline)

In [ ]:
df = load_data(DATA_PATH)
print(f"\nShape après nettoyage : {df.shape}")
print(f"Loyer médian : {df['prix_num'].median():,.0f} MAD/mois")
print(f"Prix/m² médian : {df['prix_m2'].median():.1f} MAD/m²/mois")


In [ ]:
# Distributions après nettoyage
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Distributions après nettoyage", fontsize=14, fontweight="bold")

axes[0,0].hist(df["prix_num"], bins=50, color="#2563eb", alpha=0.8, edgecolor="white")
axes[0,0].set_title("Loyer mensuel (MAD)"); axes[0,0].set_xlabel("MAD/mois")

axes[0,1].hist(np.log(df["prix_num"]), bins=50, color="#7c3aed", alpha=0.8, edgecolor="white")
axes[0,1].set_title("log(Loyer)"); axes[0,1].set_xlabel("log(MAD)")

axes[0,2].hist(df["surface_num"], bins=50, color="#10b981", alpha=0.8, edgecolor="white")
axes[0,2].set_title("Surface (m²)"); axes[0,2].set_xlabel("m²")

axes[1,0].hist(df["prix_m2"], bins=50, color="#f59e0b", alpha=0.8, edgecolor="white")
axes[1,0].set_title("Prix/m²/mois (MAD)"); axes[1,0].set_xlabel("MAD/m²/mois")

df["zone_clean"].value_counts().head(10).plot(kind="barh", ax=axes[1,1], color="#2563eb")
axes[1,1].set_title("Top zones (zone_clean)"); axes[1,1].invert_yaxis()

df["type_local"].value_counts().plot(kind="bar", ax=axes[1,2], color="#10b981")
axes[1,2].set_title("Type de local"); axes[1,2].tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()


### 2.1 Loyer médian par zone

In [ ]:
zone_stats = df.groupby("zone_clean").agg(
    nb=("prix_num", "count"),
    loyer_median=("prix_num", "median"),
    pm2_median=("prix_m2", "median"),
    surface_mediane=("surface_num", "median"),
).sort_values("loyer_median", ascending=False)
print(zone_stats.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loyer médian par zone
top_zones = zone_stats[zone_stats["nb"] >= 10].sort_values("loyer_median")
axes[0].barh(top_zones.index, top_zones["loyer_median"] / 1e3, color="#2563eb", alpha=0.85)
axes[0].set_xlabel("Loyer médian (K MAD/mois)")
axes[0].set_title("Loyer médian par zone (≥10 annonces)")

# Prix/m² médian par zone
axes[1].barh(top_zones.index, top_zones["pm2_median"], color="#10b981", alpha=0.85)
axes[1].set_xlabel("Prix/m²/mois (MAD)")
axes[1].set_title("Prix/m² médian par zone")

plt.tight_layout(); plt.show()


### 2.2 Keywords NLP

In [ ]:
kw_cols = [c for c in BINARY_FEATURES if c.startswith("kw_")]
kw_sums = df[kw_cols].sum().sort_values(ascending=False)
print("Fréquence des keywords NLP :")
print(kw_sums.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
kw_sums.plot(kind="bar", ax=ax, color="#7c3aed", alpha=0.85, edgecolor="white")
ax.set_title("Fréquence keywords NLP dans les annonces")
ax.set_ylabel("Nombre d'annonces")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


### 2.3 Corrélations avec le loyer

In [ ]:
num_cols_avail = [c for c in ["log_surface", "log_surface_sq", "prix_m2", "etage", "score_local"] if c in df.columns]
corr = df[num_cols_avail + ["log_prix"]].corr()["log_prix"].drop("log_prix").sort_values(ascending=False)
print("Corrélations avec log(loyer) :")
print(corr.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
corr.plot(kind="barh", ax=ax, color=["#2563eb" if v > 0 else "#ef4444" for v in corr.values], alpha=0.85)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Corrélation avec log(loyer)")
plt.tight_layout(); plt.show()


## 3. Split train/test & Feature Engineering (sans leakage)

In [ ]:
X_train, X_test, y_train, y_test, df_train, df_test, stats = split_and_encode(df, test_size=0.2, random_state=42)
print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"\nFeatures numériques  : {len(stats['numeric_cols'])}")
print(f"Features binaires    : {len(stats['binary_cols'])}")
print(f"Features catégorielles : {len(stats['categorical_cols'])}")
print(f"\nTotal features : {len(stats['feature_cols'])}")


In [ ]:
print("Aperçu des features d'entraînement :")
X_train.describe().round(3)


## 4. Entraînement XGBoost (paramètres par défaut)

In [ ]:
pipeline_default = build_pipeline(stats)
pipeline_default = train(pipeline_default, X_train, y_train)
metrics_default = evaluate(pipeline_default, X_train, X_test, y_train, y_test)


## 5. Optimisation Hyperparamètres — Optuna

In [ ]:
best_params = tune_hyperparams(X_train, y_train, stats, n_trials=40)
print("\nMeilleurs hyperparamètres :")
for k, v in best_params.items():
    print(f"  {k}: {v}")


In [ ]:
pipeline_tuned = build_pipeline(stats, best_params)
pipeline_tuned = train(pipeline_tuned, X_train, y_train)
metrics_tuned  = evaluate(pipeline_tuned, X_train, X_test, y_train, y_test)
stats["mode"]  = metrics_tuned["mode"]


### 5.1 Comparaison — défaut vs tuné

In [ ]:
comparison = pd.DataFrame({
    "Défaut":    {k: metrics_default.get(k, float("nan")) for k in ["r2_test", "mape", "cv_mean"]},
    "Optuna":    {k: metrics_tuned.get(k, float("nan"))   for k in ["r2_test", "mape", "cv_mean"]},
}).rename(index={"r2_test": "R² test", "mape": "MAPE (%)", "cv_mean": "CV R² (mean)"})
print(comparison.round(4).to_string())


## 6. Visualisations

In [ ]:
from pathlib import Path
save_dir = Path.cwd()  # notebooks/locations/
plot_results(pipeline_tuned, X_test, y_test, metrics_tuned, save_dir=save_dir)


### 6.1 Analyse des erreurs par zone

In [ ]:
y_pred_te = pipeline_tuned.predict(X_test)
prix_reel = np.exp(y_test)
prix_pred = np.exp(y_pred_te)
pct_err   = np.abs((prix_pred - prix_reel) / prix_reel) * 100

df_err = df_test.copy()
df_err["prix_reel"]  = prix_reel
df_err["prix_pred"]  = prix_pred
df_err["pct_err"]    = pct_err

err_by_zone = df_err.groupby("zone_clean").agg(
    n=("pct_err", "count"),
    mape_zone=("pct_err", "mean"),
    loyer_med=("prix_reel", "median"),
).sort_values("mape_zone", ascending=False)
print("Erreurs par zone :")
print(err_by_zone[err_by_zone["n"] >= 3].round(1).to_string())


In [ ]:
err_by_type = df_err.groupby("type_local").agg(
    n=("pct_err", "count"),
    mape_type=("pct_err", "mean"),
).sort_values("mape_type", ascending=False)
print("Erreurs par type de local :")
print(err_by_type.round(1).to_string())


## 7. Prédictions exemples

In [ ]:
exemples = [
    {"surface_num": 50,  "zone_clean": "Guéliz",  "type_local": "bureau",     "parking": 1, "climatisation": 1,
     "titre": "Bureau climatisé Guéliz angle"},
    {"surface_num": 100, "zone_clean": "Guéliz",  "type_local": "local_comm", "parking": 0,
     "titre": "Local commercial Guéliz centre"},
    {"surface_num": 300, "zone_clean": "autre",   "type_local": "depot",
     "titre": "Dépôt entrepôt stockage route"},
    {"surface_num": 30,  "zone_clean": "m'hamid", "type_local": "local_comm",
     "titre": "Local commercial M'hamid"},
    {"surface_num": 500, "zone_clean": "autre",   "type_local": "industriel",
     "titre": "Atelier industriel périphérie"},
]

for ex in exemples:
    result = predict_price(pipeline_tuned, ex, stats)


## 8. Sauvegarde du modèle

In [ ]:
import joblib
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipeline_tuned, "stats": stats}, MODEL_PATH)
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

# Vérification chargement
loaded = joblib.load(MODEL_PATH)
test_pred = loaded["pipeline"].predict(X_test[:3])
print(f"   Vérification — 3 prédictions log : {test_pred.round(3)}")
print(f"   → Prix estimés : {np.exp(test_pred).round(0)} MAD/mois")


## 9. Résumé & Conclusion

In [ ]:
print("=" * 55)
print("  RÉSUMÉ — LOCAUX COMMERCIAUX LOCATION v1")
print("=" * 55)
print(f"  Données brutes         : {df_raw.shape[0]} annonces")
print(f"  Après nettoyage        : {df.shape[0]} annonces")
print(f"  Features totales       : {len(stats['feature_cols'])}")
print(f"  R² test (Optuna)       : {metrics_tuned['r2_test']:.3f}")
print(f"  MAPE test              : {metrics_tuned['mape']:.1f}%")
print(f"  CV R² (5-fold train)   : {metrics_tuned['cv_mean']:.3f} ± {metrics_tuned['cv_std']:.3f}")
print(f"  MAD loyer              : {metrics_tuned['mad']:,.0f} MAD/mois")
print(f"  Mode prédiction        : {stats['mode']}")
print(f"  Modèle sauvegardé      : {MODEL_PATH.name}")
print("=" * 55)
print()
print("Seuils de nettoyage appliqués :")
print("  - Prix        : 500 – 200 000 MAD/mois")
print("  - Surface     : 5 – 10 000 m²")
print("  - Prix/m²     : 5 – 5 000 MAD/m²/mois")
print("  - Quantiles   : P1 – P99 log(prix)")
print()
print("Features engineered :")
print("  - Target encoding zone (log_prix moyen)")
print("  - Surface relative par zone")
print("  - surf_x_type : écart surface vs moyenne du type")
print("  - ratio_pm2_city : pm² relatif à la ville")
print("  - NLP keywords : bureau, dépôt, salle, angle…")
